
# Buy Fee Model V9 — Pricing Decision Engine

## Objective

Convert the V8.2 matched-control Difference-in-Differences results into a practical decision-support tool for:

- +$10
- +$15
- +$20
- +$25
- +$30

For each scenario, V9 outputs:

1. **Expected RPU gain**
2. **95% confidence interval**
3. **Conversion downside probabilities**
   - P(Conversion decline > 1 pp)
   - P(Conversion decline > 2 pp)
   - P(Conversion decline > 5 pp)
4. **Volume guardrail**
   - Low
   - Moderate
   - High
5. **Historical support**
6. **Evidence strength**
   - High
   - Medium
   - Limited
7. **Corporate change-date support**
8. **Location support**

## Modeling philosophy

V9 does **not** force a single universal elasticity.

Instead:

- RPU is modeled as the strongest financial response.
- Conversion is treated as a downside-risk probability.
- Volume is treated as a guardrail/risk metric.
- +$25 and +$30 are explicitly allowed to have limited evidence.
- All confidence intervals are based on grouped bootstrap resampling by `Corporate_Change_ID` so rows from the same pricing action are not treated as fully independent.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    brier_score_loss
)

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"

DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"
SCENARIO_FILE = V82_DIR / "05_DID_SCENARIO_EVIDENCE.csv"
SUMMARY_FILE = V82_DIR / "07_V8_2_SUMMARY.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v9_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [10, 15, 20, 25, 30]

RANDOM_STATE = 42

# High-quality RPU filter
MIN_RPU_COVERAGE = 0.90

# Conversion downside thresholds in percentage points
CONVERSION_THRESHOLDS = [1, 2, 5]

# Volume guardrail thresholds on control-adjusted change
VOLUME_LOW_RISK_CUTOFF = -0.03
VOLUME_HIGH_RISK_CUTOFF = -0.10

# Evidence strength rules
HIGH_MIN_EVENTS = 100
HIGH_MIN_DATES = 10
HIGH_MIN_LOCS = 10

MED_MIN_EVENTS = 30
MED_MIN_DATES = 5
MED_MIN_LOCS = 5

# Bootstrap
BOOTSTRAP_ITERATIONS = 1000

print("=" * 80)
print("V9 INPUT CHECK")
print("=" * 80)

for p in [DID_FILE, SCENARIO_FILE, SUMMARY_FILE]:
    print(f"{p.name:<35} {p.exists()}")

missing = [p for p in [DID_FILE, SCENARIO_FILE, SUMMARY_FILE] if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required V8.2 output(s):\n"
        + "\n".join(str(x) for x in missing)
    )


V9 INPUT CHECK
04_DID_EVENT_LEVEL_RESULTS.csv      True
05_DID_SCENARIO_EVIDENCE.csv        True
07_V8_2_SUMMARY.csv                 True


## 1. Load V8.2 matched-control results

In [2]:

did = pd.read_csv(
    DID_FILE,
    low_memory=False
)

scenario_evidence = pd.read_csv(
    SCENARIO_FILE,
    low_memory=False
)

v82_summary = pd.read_csv(
    SUMMARY_FILE,
    low_memory=False
)

print("DiD event rows:", f"{len(did):,}")
print("Locations:", did["Treated_Location"].nunique())
print("Corporate change dates:", did["Corporate_Change_ID"].nunique())

display(v82_summary)


DiD event rows: 1,824
Locations: 35
Corporate change dates: 37


,Metric,Value
0,Strict_Treated_Events,1843.000000
1,Matched_Treated_Events,1824.000000
2,Match_Rate,0.989691
3,Matched_Locations,35.000000
4,Matched_Corporate_Change_Dates,37.000000


## 2. Prepare features and quality flags

In [3]:

# Normalize dates/IDs
did["Corporate_Change_ID"] = did["Corporate_Change_ID"].astype("string")

numeric_cols = [
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
    "DiD_Fee_Pass_Through",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

for c in numeric_cols:
    if c in did.columns:
        did[c] = pd.to_numeric(
            did[c],
            errors="coerce"
        )

# If detailed pre/post RPU coverage fields are available in the matched pair source,
# V8.2 may not have carried them into event_did.
# Therefore V9 uses non-null pre_rpu + non-null DiD RPU as the core quality gate,
# and optionally applies explicit coverage if a compatible field exists.

coverage_cols = [
    c for c in did.columns
    if "coverage" in c.lower()
]

if coverage_cols:
    print("Coverage-related columns found:", coverage_cols)
else:
    print("No event-level coverage field found in V8.2 output.")
    print("RPU quality will rely on non-null matched-control RPU observations.")

did["RPU_Quality_Eligible"] = (
    did["DiD_RPU_Change_Dollar"].notna()
    & did["pre_rpu"].notna()
    & did["BUYFEE_CHANGE_AMOUNT"].gt(0)
)

print("\nRPU quality eligible:", f"{did['RPU_Quality_Eligible'].sum():,}")


No event-level coverage field found in V8.2 output.
RPU quality will rely on non-null matched-control RPU observations.

RPU quality eligible: 1,806


## 3. Helper functions

In [4]:

def make_ohe():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True
        )

def build_regression_pipeline(model, X):
    num = [
        c for c in X.columns
        if pd.api.types.is_numeric_dtype(X[c])
    ]
    cat = [c for c in X.columns if c not in num]

    prep = ColumnTransformer([
        (
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler())
            ]),
            num
        ),
        (
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("ohe", make_ohe())
            ]),
            cat
        )
    ])

    return Pipeline([
        ("prep", prep),
        ("model", model)
    ])

def build_classifier_pipeline(model, X):
    return build_regression_pipeline(model, X)

def evidence_strength(n_events, n_dates, n_locs):
    if (
        n_events >= HIGH_MIN_EVENTS
        and n_dates >= HIGH_MIN_DATES
        and n_locs >= HIGH_MIN_LOCS
    ):
        return "High"

    if (
        n_events >= MED_MIN_EVENTS
        and n_dates >= MED_MIN_DATES
        and n_locs >= MED_MIN_LOCS
    ):
        return "Medium"

    return "Limited"

def volume_guardrail_from_distribution(series):
    s = pd.to_numeric(series, errors="coerce").dropna()

    if len(s) == 0:
        return {
            "Volume_Risk": "Unknown",
            "P_Decline_3pct": np.nan,
            "P_Decline_10pct": np.nan,
            "Median_Volume_Effect": np.nan
        }

    p3 = (s <= VOLUME_LOW_RISK_CUTOFF).mean()
    p10 = (s <= VOLUME_HIGH_RISK_CUTOFF).mean()
    med = s.median()

    if p10 >= 0.35 or med <= VOLUME_HIGH_RISK_CUTOFF:
        risk = "High"
    elif p3 >= 0.40 or med <= VOLUME_LOW_RISK_CUTOFF:
        risk = "Moderate"
    else:
        risk = "Low"

    return {
        "Volume_Risk": risk,
        "P_Decline_3pct": p3,
        "P_Decline_10pct": p10,
        "Median_Volume_Effect": med
    }

def grouped_bootstrap_ci(
    df,
    value_col,
    stat="median",
    n_iter=1000,
    random_state=42
):
    d = df[
        ["Corporate_Change_ID", value_col]
    ].dropna().copy()

    if d.empty:
        return (np.nan, np.nan, np.nan)

    groups = d["Corporate_Change_ID"].unique()

    if len(groups) < 2:
        vals = d[value_col]
        point = vals.median() if stat == "median" else vals.mean()
        return (point, np.nan, np.nan)

    rng = np.random.default_rng(random_state)

    boots = []

    for _ in range(n_iter):
        sampled_groups = rng.choice(
            groups,
            size=len(groups),
            replace=True
        )

        sampled_parts = []

        for g in sampled_groups:
            sampled_parts.append(
                d[d["Corporate_Change_ID"].eq(g)]
            )

        b = pd.concat(
            sampled_parts,
            ignore_index=True
        )

        vals = b[value_col]

        if stat == "median":
            boots.append(vals.median())
        else:
            boots.append(vals.mean())

    point = (
        d[value_col].median()
        if stat == "median"
        else d[value_col].mean()
    )

    return (
        point,
        np.nanpercentile(boots, 2.5),
        np.nanpercentile(boots, 97.5)
    )


## 4. Direct historical scenario estimates with grouped confidence intervals

In [5]:

historical_rows = []

for fee in SCENARIOS:

    d = did[
        did["BUYFEE_CHANGE_AMOUNT"].between(
            fee - 2.5,
            fee + 2.5
        )
    ].copy()

    rpu_d = d[
        d["RPU_Quality_Eligible"]
    ].copy()

    rpu_point, rpu_lo, rpu_hi = grouped_bootstrap_ci(
        rpu_d,
        "DiD_RPU_Change_Dollar",
        stat="median",
        n_iter=BOOTSTRAP_ITERATIONS,
        random_state=RANDOM_STATE + fee
    )

    volume_info = volume_guardrail_from_distribution(
        d["DiD_Volume_Change_Pct"]
    )

    conversion_probs = {}

    for threshold in CONVERSION_THRESHOLDS:
        s = pd.to_numeric(
            d["DiD_Conversion_Change_pp"],
            errors="coerce"
        ).dropna()

        conversion_probs[
            f"P_Conversion_Decline_GT_{threshold}pp"
        ] = (
            (s <= -threshold).mean()
            if len(s)
            else np.nan
        )

    n_events = len(d)
    n_dates = d["Corporate_Change_ID"].nunique()
    n_locs = d["Treated_Location"].nunique()

    historical_rows.append({
        "Fee_Increase": fee,

        "Historical_Events": n_events,
        "Unique_Locations": n_locs,
        "Corporate_Change_Dates": n_dates,

        "RPU_Eligible_Events": len(rpu_d),
        "Expected_RPU_Gain_Median": rpu_point,
        "RPU_CI_Low_95": rpu_lo,
        "RPU_CI_High_95": rpu_hi,

        **conversion_probs,

        "Volume_Risk": volume_info["Volume_Risk"],
        "P_Volume_Decline_GT_3pct": volume_info["P_Decline_3pct"],
        "P_Volume_Decline_GT_10pct": volume_info["P_Decline_10pct"],
        "Median_DiD_Volume_Effect": volume_info["Median_Volume_Effect"],

        "Evidence_Strength": evidence_strength(
            n_events,
            n_dates,
            n_locs
        )
    })

historical_scenario = pd.DataFrame(
    historical_rows
)

historical_scenario.to_csv(
    OUTPUT_DIR / "01_HISTORICAL_SCENARIO_ESTIMATES.csv",
    index=False
)

display(historical_scenario)


,Fee_Increase,Historical_Events,Unique_Locations,Corporate_Change_Dates,RPU_Eligible_Events,Expected_RPU_Gain_Median,RPU_CI_Low_95,RPU_CI_High_95,P_Conversion_Decline_GT_1pp,P_Conversion_Decline_GT_2pp,P_Conversion_Decline_GT_5pp,Volume_Risk,P_Volume_Decline_GT_3pct,P_Volume_Decline_GT_10pct,Median_DiD_Volume_Effect,Evidence_Strength
0,10,594,16,24,594,7.630647,0.130583,17.366846,0.444444,0.414141,0.326599,High,0.466330,0.388889,-0.003620,High
1,15,84,14,10,84,10.481982,7.286644,19.725929,0.428571,0.392857,0.321429,Moderate,0.404762,0.345238,0.044703,Medium
2,20,133,12,10,133,24.466728,3.153149,37.740392,0.436090,0.413534,0.315789,High,0.496241,0.406015,-0.025353,High
3,25,42,10,4,42,2.220601,-23.328829,22.817812,0.428571,0.380952,0.309524,Low,0.357143,0.309524,0.073548,Limited
4,30,60,8,4,60,2.512057,-22.996534,31.629251,0.483333,0.466667,0.300000,Moderate,0.416667,0.283333,0.036691,Limited


## 5. Train RPU gain model

In [6]:

rpu_df = did[
    did["RPU_Quality_Eligible"]
].copy()

feature_candidates = [
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

rpu_features = [
    c for c in feature_candidates
    if c in rpu_df.columns
]

rpu_df = rpu_df.dropna(
    subset=["DiD_RPU_Change_Dollar"]
).copy()

# Grouped chronological holdout: last 20% of corporate change IDs
group_order = (
    rpu_df[
        ["Corporate_Change_ID"]
    ]
    .drop_duplicates()
    ["Corporate_Change_ID"]
    .tolist()
)

split = max(1, int(len(group_order) * 0.80))

train_groups = group_order[:split]
test_groups = group_order[split:]

rpu_train = rpu_df[
    rpu_df["Corporate_Change_ID"].isin(train_groups)
].copy()

rpu_test = rpu_df[
    rpu_df["Corporate_Change_ID"].isin(test_groups)
].copy()

Xtr = rpu_train[rpu_features]
ytr = rpu_train["DiD_RPU_Change_Dollar"]

Xte = rpu_test[rpu_features]
yte = rpu_test["DiD_RPU_Change_Dollar"]

ridge = build_regression_pipeline(
    Ridge(alpha=10.0),
    Xtr
)

rf = build_regression_pipeline(
    RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=max(3, int(len(rpu_train) * 0.02)),
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    Xtr
)

ridge.fit(Xtr, ytr)
rf.fit(Xtr, ytr)

pred = (
    ridge.predict(Xte)
    + rf.predict(Xte)
) / 2

rpu_metrics = pd.DataFrame([{
    "Train_Rows": len(rpu_train),
    "Test_Rows": len(rpu_test),
    "Train_Change_Dates": len(train_groups),
    "Test_Change_Dates": len(test_groups),
    "MAE": mean_absolute_error(yte, pred),
    "R2": r2_score(yte, pred) if len(yte) > 1 else np.nan,
    "Direction_Accuracy": np.mean(
        np.sign(pred) == np.sign(yte)
    )
}])

rpu_metrics.to_csv(
    OUTPUT_DIR / "02_RPU_MODEL_VALIDATION.csv",
    index=False
)

display(rpu_metrics)

# Refit full
ridge_full = build_regression_pipeline(
    Ridge(alpha=10.0),
    rpu_df[rpu_features]
)

rf_full = build_regression_pipeline(
    RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=max(3, int(len(rpu_df) * 0.02)),
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    rpu_df[rpu_features]
)

ridge_full.fit(
    rpu_df[rpu_features],
    rpu_df["DiD_RPU_Change_Dollar"]
)

rf_full.fit(
    rpu_df[rpu_features],
    rpu_df["DiD_RPU_Change_Dollar"]
)


,Train_Rows,Test_Rows,Train_Change_Dates,Test_Change_Dates,MAE,R2,Direction_Accuracy
0,1705,101,28,8,22.174751,0.05684,0.49505


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['BUYFEE_CHANGE_AMOUNT','PREVIOUS_BUYFEE','FEE_CHANGE_PCT',..., 'pre_conversion','pre_rpu','pre_avg_saleamt']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed t

## 6. Train conversion-risk classifiers

In [7]:

conversion_features = [
    c for c in feature_candidates
    if c in did.columns
]

conversion_models = {}
conversion_validation_rows = []

for threshold in CONVERSION_THRESHOLDS:

    d = did.dropna(
        subset=["DiD_Conversion_Change_pp"]
    ).copy()

    target_col = f"Conversion_Down_GT_{threshold}pp"

    d[target_col] = (
        d["DiD_Conversion_Change_pp"] <= -threshold
    ).astype(int)

    groups = d["Corporate_Change_ID"].drop_duplicates().tolist()

    split = max(1, int(len(groups) * 0.80))

    train_groups = groups[:split]
    test_groups = groups[split:]

    train = d[
        d["Corporate_Change_ID"].isin(train_groups)
    ]

    test = d[
        d["Corporate_Change_ID"].isin(test_groups)
    ]

    Xtr = train[conversion_features]
    ytr = train[target_col]

    Xte = test[conversion_features]
    yte = test[target_col]

    if ytr.nunique() < 2 or yte.nunique() < 2:
        conversion_validation_rows.append({
            "Threshold_pp": threshold,
            "Status": "Insufficient class variation"
        })
        continue

    logit = build_classifier_pipeline(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ),
        Xtr
    )

    rf_clf = build_classifier_pipeline(
        RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=max(3, int(len(train) * 0.02)),
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        Xtr
    )

    logit.fit(Xtr, ytr)
    rf_clf.fit(Xtr, ytr)

    p = (
        logit.predict_proba(Xte)[:, 1]
        + rf_clf.predict_proba(Xte)[:, 1]
    ) / 2

    conversion_validation_rows.append({
        "Threshold_pp": threshold,
        "Status": "OK",
        "Train_Rows": len(train),
        "Test_Rows": len(test),
        "ROC_AUC": roc_auc_score(yte, p),
        "Brier_Score": brier_score_loss(yte, p),
        "Observed_Rate_Test": yte.mean(),
        "Predicted_Rate_Test": p.mean()
    })

    # Refit full
    logit_full = build_classifier_pipeline(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ),
        d[conversion_features]
    )

    rf_full_clf = build_classifier_pipeline(
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=max(3, int(len(d) * 0.02)),
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        d[conversion_features]
    )

    logit_full.fit(
        d[conversion_features],
        d[target_col]
    )

    rf_full_clf.fit(
        d[conversion_features],
        d[target_col]
    )

    conversion_models[threshold] = {
        "logit": logit_full,
        "rf": rf_full_clf,
        "features": conversion_features
    }

conversion_validation = pd.DataFrame(
    conversion_validation_rows
)

conversion_validation.to_csv(
    OUTPUT_DIR / "03_CONVERSION_RISK_VALIDATION.csv",
    index=False
)

display(conversion_validation)


,Threshold_pp,Status,Train_Rows,Test_Rows,ROC_AUC,Brier_Score,Observed_Rate_Test,Predicted_Rate_Test
0,1,OK,1723,101,0.516883,0.277615,0.346535,0.570653
1,2,OK,1723,101,0.531130,0.287444,0.287129,0.575596
2,5,OK,1723,101,0.577664,0.301260,0.188119,0.574819


## 7. Build +$10 / +$15 / +$20 / +$25 / +$30 decision table

In [8]:

# Use median historical pre-condition profile as a portfolio-level reference scenario.
base_reference = {}

for c in rpu_features:
    if c in [
        "Treated_Location",
        "Treated_Lane",
        "Treated_Price_Tier"
    ]:
        mode = did[c].mode(dropna=True)
        base_reference[c] = mode.iloc[0] if len(mode) else "UNKNOWN"
    else:
        base_reference[c] = pd.to_numeric(
            did[c],
            errors="coerce"
        ).median()

decision_rows = []

for fee in SCENARIOS:

    scenario_row = dict(base_reference)

    scenario_row["BUYFEE_CHANGE_AMOUNT"] = fee

    prev_fee = scenario_row.get(
        "PREVIOUS_BUYFEE",
        np.nan
    )

    scenario_row["FEE_CHANGE_PCT"] = (
        fee / prev_fee
        if pd.notna(prev_fee)
        and prev_fee != 0
        else np.nan
    )

    X_scenario = pd.DataFrame(
        [scenario_row]
    )

    # ML RPU prediction
    model_rpu = (
        ridge_full.predict(
            X_scenario[rpu_features]
        )[0]
        +
        rf_full.predict(
            X_scenario[rpu_features]
        )[0]
    ) / 2

    hist = historical_scenario[
        historical_scenario["Fee_Increase"].eq(fee)
    ].iloc[0]

    # Blend direct history and ML only when historical data exists
    if pd.notna(
        hist["Expected_RPU_Gain_Median"]
    ):
        expected_rpu = (
            0.60 * hist["Expected_RPU_Gain_Median"]
            +
            0.40 * model_rpu
        )
    else:
        expected_rpu = model_rpu

    conversion_probs = {}

    for threshold in CONVERSION_THRESHOLDS:

        model_info = conversion_models.get(threshold)

        if model_info is None:
            conversion_probs[
                f"P_Conversion_Decline_GT_{threshold}pp"
            ] = hist[
                f"P_Conversion_Decline_GT_{threshold}pp"
            ]
            continue

        features = model_info["features"]

        # Make sure all needed fields exist
        conv_row = {}

        for c in features:
            if c in scenario_row:
                conv_row[c] = scenario_row[c]
            elif c in did.columns:
                if pd.api.types.is_numeric_dtype(did[c]):
                    conv_row[c] = pd.to_numeric(
                        did[c],
                        errors="coerce"
                    ).median()
                else:
                    mode = did[c].mode(dropna=True)
                    conv_row[c] = (
                        mode.iloc[0]
                        if len(mode)
                        else "UNKNOWN"
                    )

        Xc = pd.DataFrame([conv_row])

        p = (
            model_info["logit"].predict_proba(
                Xc[features]
            )[:, 1][0]
            +
            model_info["rf"].predict_proba(
                Xc[features]
            )[:, 1][0]
        ) / 2

        # Blend with direct empirical risk
        empirical = hist[
            f"P_Conversion_Decline_GT_{threshold}pp"
        ]

        if pd.notna(empirical):
            p = 0.50 * p + 0.50 * empirical

        conversion_probs[
            f"P_Conversion_Decline_GT_{threshold}pp"
        ] = p

    decision_rows.append({
        "Fee_Increase": fee,

        "Expected_RPU_Gain": expected_rpu,

        "Historical_RPU_Median": hist[
            "Expected_RPU_Gain_Median"
        ],

        "Historical_RPU_CI_Low_95": hist[
            "RPU_CI_Low_95"
        ],

        "Historical_RPU_CI_High_95": hist[
            "RPU_CI_High_95"
        ],

        **conversion_probs,

        "Volume_Risk": hist["Volume_Risk"],

        "P_Volume_Decline_GT_3pct": hist[
            "P_Volume_Decline_GT_3pct"
        ],

        "P_Volume_Decline_GT_10pct": hist[
            "P_Volume_Decline_GT_10pct"
        ],

        "Historical_Events": hist[
            "Historical_Events"
        ],

        "Corporate_Change_Dates": hist[
            "Corporate_Change_Dates"
        ],

        "Unique_Locations": hist[
            "Unique_Locations"
        ],

        "Evidence_Strength": hist[
            "Evidence_Strength"
        ],
    })

decision_table = pd.DataFrame(
    decision_rows
)

decision_table.to_csv(
    OUTPUT_DIR / "04_V9_DECISION_TABLE.csv",
    index=False
)

display(decision_table)


,Fee_Increase,Expected_RPU_Gain,Historical_RPU_Median,Historical_RPU_CI_Low_95,Historical_RPU_CI_High_95,P_Conversion_Decline_GT_1pp,P_Conversion_Decline_GT_2pp,P_Conversion_Decline_GT_5pp,Volume_Risk,P_Volume_Decline_GT_3pct,P_Volume_Decline_GT_10pct,Historical_Events,Corporate_Change_Dates,Unique_Locations,Evidence_Strength
0,10,5.275208,7.630647,0.130583,17.366846,0.528775,0.513420,0.469575,High,0.466330,0.388889,594,24,16,High
1,15,7.067430,10.481982,7.286644,19.725929,0.522969,0.504040,0.468044,Moderate,0.404762,0.345238,84,10,14,Medium
2,20,15.469242,24.466728,3.153149,37.740392,0.527427,0.514665,0.464451,High,0.496241,0.406015,133,10,12,High
3,25,1.646721,2.220601,-23.328829,22.817812,0.525478,0.499971,0.463892,Low,0.357143,0.309524,42,4,10,Limited
4,30,1.615425,2.512057,-22.996534,31.629251,0.554714,0.544805,0.461252,Moderate,0.416667,0.283333,60,4,8,Limited


## 8. Add interpretation bands

In [9]:

def conversion_risk_band(p2):
    if pd.isna(p2):
        return "Unknown"

    if p2 < 0.20:
        return "Low"

    if p2 < 0.40:
        return "Moderate"

    return "High"

decision_table["Conversion_Risk"] = decision_table[
    "P_Conversion_Decline_GT_2pp"
].apply(conversion_risk_band)

decision_table["RPU_Confidence"] = np.select(
    [
        decision_table["Evidence_Strength"].eq("High"),
        decision_table["Evidence_Strength"].eq("Medium"),
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

decision_table["Scenario_Use"] = np.where(
    decision_table["Evidence_Strength"].eq("Limited"),
    "Use cautiously / limited direct evidence",
    "Suitable for decision-support review"
)

decision_table.to_csv(
    OUTPUT_DIR / "05_V9_DECISION_TABLE_WITH_BANDS.csv",
    index=False
)

display(
    decision_table[
        [
            "Fee_Increase",
            "Expected_RPU_Gain",
            "Historical_RPU_CI_Low_95",
            "Historical_RPU_CI_High_95",
            "Conversion_Risk",
            "P_Conversion_Decline_GT_2pp",
            "Volume_Risk",
            "Evidence_Strength",
            "Scenario_Use"
        ]
    ]
)


,Fee_Increase,Expected_RPU_Gain,Historical_RPU_CI_Low_95,Historical_RPU_CI_High_95,Conversion_Risk,P_Conversion_Decline_GT_2pp,Volume_Risk,Evidence_Strength,Scenario_Use
0,10,5.275208,0.130583,17.366846,High,0.513420,High,High,Suitable for decision-support review
1,15,7.067430,7.286644,19.725929,High,0.504040,Moderate,Medium,Suitable for decision-support review
2,20,15.469242,3.153149,37.740392,High,0.514665,High,High,Suitable for decision-support review
3,25,1.646721,-23.328829,22.817812,High,0.499971,Low,Limited,Use cautiously / limited direct evidence
4,30,1.615425,-22.996534,31.629251,High,0.544805,Moderate,Limited,Use cautiously / limited direct evidence


## 9. Scenario-level readiness

In [10]:

readiness_rows = []

for _, row in decision_table.iterrows():

    if row["Evidence_Strength"] == "High":
        readiness = "GO"

    elif row["Evidence_Strength"] == "Medium":
        readiness = "GO WITH CAUTION"

    else:
        readiness = "RESEARCH ONLY"

    readiness_rows.append({
        "Fee_Increase": row["Fee_Increase"],
        "Readiness": readiness,
        "Expected_RPU_Gain": row["Expected_RPU_Gain"],
        "Conversion_Risk": row["Conversion_Risk"],
        "Volume_Risk": row["Volume_Risk"],
        "Evidence_Strength": row["Evidence_Strength"],
        "Corporate_Change_Dates": row["Corporate_Change_Dates"],
        "Historical_Events": row["Historical_Events"],
    })

scenario_readiness = pd.DataFrame(
    readiness_rows
)

scenario_readiness.to_csv(
    OUTPUT_DIR / "06_V9_SCENARIO_READINESS.csv",
    index=False
)

display(scenario_readiness)


,Fee_Increase,Readiness,Expected_RPU_Gain,Conversion_Risk,Volume_Risk,Evidence_Strength,Corporate_Change_Dates,Historical_Events
0,10,GO,5.275208,High,High,High,24,594
1,15,GO WITH CAUTION,7.067430,High,Moderate,Medium,10,84
2,20,GO,15.469242,High,High,High,10,133
3,25,RESEARCH ONLY,1.646721,High,Low,Limited,4,42
4,30,RESEARCH ONLY,1.615425,High,Moderate,Limited,4,60


## 10. Final executive output

In [11]:

executive = decision_table[
    [
        "Fee_Increase",
        "Expected_RPU_Gain",
        "Historical_RPU_CI_Low_95",
        "Historical_RPU_CI_High_95",

        "P_Conversion_Decline_GT_1pp",
        "P_Conversion_Decline_GT_2pp",
        "P_Conversion_Decline_GT_5pp",
        "Conversion_Risk",

        "P_Volume_Decline_GT_3pct",
        "P_Volume_Decline_GT_10pct",
        "Volume_Risk",

        "Historical_Events",
        "Corporate_Change_Dates",
        "Unique_Locations",

        "Evidence_Strength",
        "RPU_Confidence",
        "Scenario_Use"
    ]
].copy()

executive = executive.merge(
    scenario_readiness[
        ["Fee_Increase", "Readiness"]
    ],
    on="Fee_Increase",
    how="left"
)

executive.to_csv(
    OUTPUT_DIR / "07_V9_EXECUTIVE_SCENARIO_OUTPUT.csv",
    index=False
)

display(executive)


,Fee_Increase,Expected_RPU_Gain,Historical_RPU_CI_Low_95,Historical_RPU_CI_High_95,P_Conversion_Decline_GT_1pp,P_Conversion_Decline_GT_2pp,P_Conversion_Decline_GT_5pp,Conversion_Risk,P_Volume_Decline_GT_3pct,P_Volume_Decline_GT_10pct,Volume_Risk,Historical_Events,Corporate_Change_Dates,Unique_Locations,Evidence_Strength,RPU_Confidence,Scenario_Use,Readiness
0,10,5.275208,0.130583,17.366846,0.528775,0.513420,0.469575,High,0.466330,0.388889,High,594,24,16,High,High,Suitable for decision-support review,GO
1,15,7.067430,7.286644,19.725929,0.522969,0.504040,0.468044,High,0.404762,0.345238,Moderate,84,10,14,Medium,Medium,Suitable for decision-support review,GO WITH CAUTION
2,20,15.469242,3.153149,37.740392,0.527427,0.514665,0.464451,High,0.496241,0.406015,High,133,10,12,High,High,Suitable for decision-support review,GO
3,25,1.646721,-23.328829,22.817812,0.525478,0.499971,0.463892,High,0.357143,0.309524,Low,42,4,10,Limited,Low,Use cautiously / limited direct evidence,RESEARCH ONLY
4,30,1.615425,-22.996534,31.629251,0.554714,0.544805,0.461252,High,0.416667,0.283333,Moderate,60,4,8,Limited,Low,Use cautiously / limited direct evidence,RESEARCH ONLY



## Files to share back

Please send these after running V9:

1. `07_V9_EXECUTIVE_SCENARIO_OUTPUT.csv`
2. `06_V9_SCENARIO_READINESS.csv`
3. `05_V9_DECISION_TABLE_WITH_BANDS.csv`
4. `02_RPU_MODEL_VALIDATION.csv`
5. `03_CONVERSION_RISK_VALIDATION.csv`
6. `01_HISTORICAL_SCENARIO_ESTIMATES.csv`

## What we will evaluate next

We will check:

- whether expected RPU gain is sensible for each fee increase,
- whether conversion-risk probabilities are calibrated,
- whether volume guardrails are stable,
- whether +$25 / +$30 should remain research-only,
- whether buyer composition should be added as a V9.1 enhancement,
- and whether the scenario table is ready for an Excel/Pricing decision tool.
